# Notebook 07: back-translation augmentation for the two minority classes

Feature Request and UX Feedback are the two weakest classes, and they appear to be weak
for different reasons: Feature Request through scarcity (130 of 1,540
training rows, under a quarter of Positive Praise's count), UX Feedback through an
ambiguous boundary with Bug Report.

Those two explanations predict opposite responses to synthetic data, and that is what
this notebook tests. Both classes are augmented by back-translation (English -> French
-> English) and re-evaluated on the untouched test set. If scarcity is the binding
constraint, more data should help; if ambiguity is, paraphrasing an already-ambiguous
review should blur it further.

In [1]:
%pip install transformers torch scikit-learn pandas numpy tqdm sentencepiece sacremoses


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import copy
import json
import os
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup,
    MarianTokenizer, MarianMTModel,
)
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)


if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


## Step 1: Load data and pull out the Feature Request + UX Feedback rows

In [ ]:
train_df = pd.read_csv('../data/processed/train.csv')
val_df   = pd.read_csv('../data/processed/val.csv')
test_df  = pd.read_csv('../data/processed/test.csv')

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print()
print('Category distribution in train (before augmentation):')
print(train_df['category_name'].value_counts())

CATEGORY_NAMES = ['Bug Report', 'Feature Request', 'UX Feedback', 'Positive Praise']

# Feature Request: augment all of it (smallest class by far, 130 rows).
# UX Feedback: augment a capped 200-row random subsample rather than all 307,
# because doubling the whole class would take it to 614 — past both Bug Report
# (520) and Positive Praise (583) — and load one class with paraphrase-heavy
# synthetic text. Capping brings it to 507, roughly in line with Bug Report.
fr_rows = train_df[train_df['category_name'] == 'Feature Request'].copy()
ux_rows = train_df[train_df['category_name'] == 'UX Feedback'].sample(
    n=min(200, (train_df['category_name'] == 'UX Feedback').sum()), random_state=42
).copy()

print(f'\nFeature Request rows to augment: {len(fr_rows)}')
print(f'UX Feedback rows to augment: {len(ux_rows)}')

Train: 1540 | Val: 330 | Test: 330

Category distribution in train (before augmentation):
category_name
Positive Praise    583
Bug Report         520
UX Feedback        307
Feature Request    130
Name: count, dtype: int64

Feature Request rows to augment: 130
UX Feedback rows to augment: 200


## Step 2: Back-translation (English -> French -> English)

Translating to French and back gives a paraphrase that keeps roughly the same
meaning but different wording. Doing this with local MarianMT models instead of an
API keeps it free, and avoids an LLM possibly rewriting the review enough to drift
its label.

In [4]:
tok_fwd = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-en-fr')
model_fwd = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-en-fr').to(device)
tok_bwd = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-fr-en')
model_bwd = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-fr-en').to(device)
print('Translation models loaded')


def back_translate(texts, batch_size=8):
    """EN -> FR -> EN, batched, returns a list of paraphrased strings."""
    texts = [str(t) for t in texts]
    french = []
    for i in tqdm(range(0, len(texts), batch_size), desc='EN->FR'):
        batch = texts[i:i + batch_size]
        enc = tok_fwd(batch, return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            out = model_fwd.generate(**enc, max_length=128)
        french.extend(tok_fwd.batch_decode(out, skip_special_tokens=True))

    english = []
    for i in tqdm(range(0, len(french), batch_size), desc='FR->EN'):
        batch = french[i:i + batch_size]
        enc = tok_bwd(batch, return_tensors='pt', padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            out = model_bwd.generate(**enc, max_length=128)
        english.extend(tok_bwd.batch_decode(out, skip_special_tokens=True))

    return english

Loading weights: 100%|██████████| 256/256 [00:00<00:00, 7256.83it/s]


Translation models loaded


In [5]:
print('Back-translating Feature Request rows...')
fr_paraphrased = back_translate(fr_rows['clean_text'].tolist())

print('\nBack-translating UX Feedback rows...')
ux_paraphrased = back_translate(ux_rows['clean_text'].tolist())

fr_aug = fr_rows.copy()
fr_aug['clean_text'] = fr_paraphrased
fr_aug['review_id'] = fr_aug['review_id'].astype(str) + '-augmented'

ux_aug = ux_rows.copy()
ux_aug['clean_text'] = ux_paraphrased
ux_aug['review_id'] = ux_aug['review_id'].astype(str) + '-augmented'

train_augmented_df = pd.concat([train_df, fr_aug, ux_aug], ignore_index=True)

print('\nCategory distribution in train (after augmentation):')
print(train_augmented_df['category_name'].value_counts())
print(f'\nTotal train rows: {len(train_augmented_df)} (was {len(train_df)})')

os.makedirs('../data/processed', exist_ok=True)
train_augmented_df.to_csv('../data/processed/train_augmented.csv', index=False)
print('Saved to ../data/processed/train_augmented.csv')

Back-translating Feature Request rows...


FR->EN: 100%|██████████| 17/17 [00:33<00:00,  1.96s/it]



Back-translating UX Feedback rows...


FR->EN: 100%|██████████| 25/25 [00:46<00:00,  1.86s/it]


Category distribution in train (after augmentation):
category_name
Positive Praise    583
Bug Report         520
UX Feedback        507
Feature Request    260
Name: count, dtype: int64

Total train rows: 1870 (was 1540)
Saved to ../data/processed/train_augmented.csv


## Step 3: Re-train BERT on the augmented train set

Same setup as notebook 04 (focal loss, warmup, early stopping, no-decay
bias/LayerNorm), just with the augmented training data. Val and test sets are
unchanged, so the comparison against notebook 04's baseline is fair.

In [6]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN = 128
BATCH_SIZE = 16
BEST_LR = 2e-5  # won every previous sweep for the category task


class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]), max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
        }


class FocalLoss(nn.Module):
    def __init__(self, class_weights, gamma=2.0):
        super().__init__()
        self.class_weights = class_weights
        self.gamma = gamma

    def forward(self, logits, targets):
        # unweighted CE recovers the true p_t, so the focal term is independent of the class weight
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_term = (1 - pt) ** self.gamma
        # class weight applied separately as the focal alpha, so the two corrections compose, not multiply
        alpha = self.class_weights[targets]
        return (alpha * focal_term * ce_loss).mean()


def evaluate_epoch(model, val_loader, device):
    model.eval()
    total_val_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            outputs = model(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
            labels = batch['label'].to(device)
            loss = F.cross_entropy(outputs.logits, labels)
            total_val_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch['label'].numpy())
    avg_val_loss = total_val_loss / len(val_loader)
    report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)
    return avg_val_loss, report['macro avg']['f1-score'], all_preds, all_labels


train_dataset = ReviewDataset(train_augmented_df['clean_text'].tolist(), train_augmented_df['category_label'].tolist(), tokenizer, MAX_LEN)
val_dataset   = ReviewDataset(val_df['clean_text'].tolist(), val_df['category_label'].tolist(), tokenizer, MAX_LEN)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4).to(device)

class_weights = compute_class_weight('balanced', classes=np.arange(4), y=train_augmented_df['category_label'].values)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f'Class weights (augmented train set): {class_weights.cpu().numpy().round(3)}')
loss_fn = FocalLoss(class_weights, gamma=2.0)

no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
]
optimizer = AdamW(optimizer_grouped_parameters, lr=BEST_LR, eps=1e-8)
MAX_EPOCHS, PATIENCE = 6, 2
total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

best_macro_f1, best_state, epochs_without_improvement = -1, None, 0
for epoch in range(MAX_EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{MAX_EPOCHS} [Train]'):
        optimizer.zero_grad()
        outputs = model(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
        loss = loss_fn(outputs.logits, batch['label'].to(device))
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_loss / len(train_loader)
    avg_val_loss, val_macro_f1, _, _ = evaluate_epoch(model, val_loader, device)
    print(f'  Epoch {epoch+1}: Train Loss={avg_train_loss:.4f} | Val Loss={avg_val_loss:.4f} | Val Macro F1={val_macro_f1:.4f}')

    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}')
            break

model.load_state_dict(best_state)
print(f'\nBest val macro F1 (augmented model): {best_macro_f1:.4f}')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6509.64it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Class weights (augmented train set): [0.899 1.798 0.922 0.802]


Epoch 1/6 [Train]: 100%|██████████| 117/117 [01:39<00:00,  1.18it/s]


  Epoch 1: Train Loss=0.6765 | Val Loss=0.9052 | Val Macro F1=0.5889


Epoch 2/6 [Train]: 100%|██████████| 117/117 [01:38<00:00,  1.19it/s]


...

  Epoch 3: Train Loss=0.1636 | Val Loss=0.6817 | Val Macro F1=0.6856


Epoch 4/6 [Train]: 100%|██████████| 117/117 [01:40<00:00,  1.16it/s]


  Epoch 4: Train Loss=0.0604 | Val Loss=0.7008 | Val Macro F1=0.6585


Epoch 5/6 [Train]: 100%|██████████| 117/117 [01:44<00:00,  1.12it/s]


  Epoch 5: Train Loss=0.0291 | Val Loss=0.7273 | Val Macro F1=0.6578
  Early stopping at epoch 5

Best val macro F1 (augmented model): 0.6856


## Step 4: Evaluate on the held-out test set (unchanged)

In [7]:
def evaluate_bert(model, test_df, label_col, label_names, tokenizer, device, batch_size=16, max_len=128):
    test_dataset = ReviewDataset(test_df['clean_text'].tolist(), test_df[label_col].tolist(), tokenizer, max_len)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Evaluating'):
            outputs = model(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device))
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch['label'].numpy())
    report = classification_report(all_labels, all_preds, target_names=label_names, output_dict=True, zero_division=0)
    print(classification_report(all_labels, all_preds, target_names=label_names, zero_division=0))
    macro_f1 = report['macro avg']['f1-score']
    print(f'Accuracy: {accuracy_score(all_labels, all_preds):.4f}')
    print(f'Macro F1: {macro_f1:.4f}')
    return all_preds, all_labels, macro_f1, report


aug_preds, aug_labels, aug_macro_f1, aug_report = evaluate_bert(
    model, test_df, 'category_label', CATEGORY_NAMES, tokenizer, device
)

Evaluating: 100%|██████████| 21/21 [00:04<00:00,  4.91it/s]

                 precision    recall  f1-score   support

     Bug Report       0.66      0.79      0.72       112
Feature Request       0.68      0.63      0.65        27
    UX Feedback       0.62      0.44      0.51        66
Positive Praise       0.82      0.81      0.81       125

       accuracy                           0.72       330
      macro avg       0.69      0.67      0.68       330
   weighted avg       0.71      0.72      0.71       330

Accuracy: 0.7152
Macro F1: 0.6756


## Step 5: Compare against notebook 04 and decide whether to keep this model

In [8]:
with open('../reports/baseline_results.json') as f:
    baseline_results = json.load(f)

baseline_macro_f1 = baseline_results['bert_category_macro_f1']
baseline_acc = baseline_results['bert_category_accuracy']
aug_acc = accuracy_score(aug_labels, aug_preds)

print('=== Before vs after back-translation augmentation (Feature Request + UX Feedback) ===')
print(f'Baseline (notebook 04):    accuracy={baseline_acc:.4f}  macro F1={baseline_macro_f1:.4f}')
print(f'Augmented (this notebook): accuracy={aug_acc:.4f}  macro F1={aug_macro_f1:.4f}')
print(f'Delta: {aug_macro_f1 - baseline_macro_f1:+.4f} macro F1')

print('\nPer-class F1, baseline vs augmented:')
for cls in CATEGORY_NAMES:
    base_f1 = None  # per-class baseline F1 is not stored in baseline_results.json; see notebook 04's classification report
    aug_f1 = aug_report[cls]['f1-score']
    print(f'  {cls:<18} augmented F1={aug_f1:.4f}')

IMPROVED = bool(aug_macro_f1 > baseline_macro_f1)
print(f'\nIMPROVED = {IMPROVED}')

=== Before vs after back-translation augmentation (Feature Request + UX Feedback) ===
Baseline (notebook 04):    accuracy=0.7212  macro F1=0.6754
Augmented (this notebook): accuracy=0.7152  macro F1=0.6756
Delta: +0.0002 macro F1

Per-class F1, baseline vs augmented:
  Bug Report         augmented F1=0.7206
  Feature Request    augmented F1=0.6538
  UX Feedback        augmented F1=0.5133
  Positive Praise    augmented F1=0.8145

IMPROVED = True


In [9]:
os.makedirs('../reports', exist_ok=True)

augmentation_results = {
    'baseline_bert_category_accuracy': baseline_acc,
    'baseline_bert_category_macro_f1': baseline_macro_f1,
    'augmented_bert_category_accuracy': float(aug_acc),
    'augmented_bert_category_macro_f1': float(aug_macro_f1),
    'augmented_per_class_f1': {cls: aug_report[cls]['f1-score'] for cls in CATEGORY_NAMES},
    'improved': IMPROVED,
    'augmentation_scope': 'Feature Request (all 130 rows) + UX Feedback (200-row random subsample)',
}
with open('../reports/augmentation_results.json', 'w') as f:
    json.dump(augmentation_results, f, indent=2)
print('Saved to ../reports/augmentation_results.json:', augmentation_results)

if IMPROVED:
    os.makedirs('../models/bert_category_augmented', exist_ok=True)
    model.save_pretrained('../models/bert_category_augmented')
    tokenizer.save_pretrained('../models/bert_category_augmented')
    print('Augmented model improved on the baseline. Saved to ../models/bert_category_augmented')
    print('(Not automatically promoted over ../models/bert_category. Decide manually whether to replace it.)')
else:
    print('Augmented model did not improve on the baseline, so it was not saved. The baseline model in ../models/bert_category stands.')

Saved to ../reports/augmentation_results.json: {'baseline_bert_category_accuracy': 0.7212121212121212, 'baseline_bert_category_macro_f1': 0.6753887803220903, 'augmented_bert_category_accuracy': 0.7151515151515152, 'augmented_bert_category_macro_f1': 0.6755710981102375, 'augmented_per_class_f1': {'Bug Report': 0.7206477732793523, 'Feature Request': 0.6538461538461539, 'UX Feedback': 0.5132743362831859, 'Positive Praise': 0.8145161290322581}, 'improved': True, 'augmentation_scope': 'Feature Request (all 130 rows) + UX Feedback (200-row random subsample)'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.43it/s]

Augmented model improved on the baseline. Saved to ../models/bert_category_augmented
(Not automatically promoted over ../models/bert_category. Decide manually whether to replace it.)
